# Explaination

Why use ANN-based weightage instead of manually selecting combinations?

Manual combination testing gives separate scores, but it does not learn a general rule for how detector, recognizer, and scale influence performance together. ANN-based weighting uses known ground-truth identities in scene images to learn this relationship directly from data.

Key benefits of learned weightage:

- **Data-driven decisions:** The system learns from measured precision/recall/F1 outcomes, not intuition.
- **Captures interactions:** Detector, recognizer, and scale interact; ANN can model these dependencies better than manual weighting.
- **Reduced bias and better consistency:** Learned weights are reproducible for the same training data, unlike subjective manual tuning.
- **Interpretable importance:** Weightages indicate which detector and recognizer choices contribute most to identification performance.
- **Scales to larger search spaces:** As the number of models and settings grows, manual comparison becomes difficult, while the ANN approach remains structured and maintainable.

Why set weightages?

Weightages quantify how strongly each pipeline component contributes to final recognition quality. This helps justify model choices scientifically, prioritize optimization effort, and provide a clear, defensible methodology in reports.

# Model Combinations (ANN Weighting)

This notebook evaluates DeepFace detector + recognizer + scale combinations and then trains a **simple ANN** to learn which combinations perform best.

It uses known scene-image identities (from filename tokens) as supervision and outputs learned weightage for detector, recognizer, and scale.

In [ ]:
# -----------------------------
# Imports
# -----------------------------
# itertools: Cartesian products for all detector/recognizer/scale combinations
# re: parse identity letters from scene filenames
# Path: platform-safe file and folder handling
import itertools
import re
from pathlib import Path

# OpenCV for image read/resize
import cv2
# NumPy for vectors, ANN, and math
import numpy as np
# Pandas for tabular tracking of results
import pandas as pd
# DeepFace for detection + embedding extraction
from deepface import DeepFace
# Progress bars for long loops
from tqdm.auto import tqdm

# -----------------------------
# Global configuration
# -----------------------------
# Notebook assumes it is run from project root.
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "open_data_set"

# Candidate detector backends to compare.
DETECTORS = ["opencv", "mtcnn", "retinaface", "ssd", "mediapipe"]
# Candidate recognizer models to compare.
RECOGNIZERS = ["ArcFace", "Facenet512", "SFace", "GhostFaceNet", "Dlib"]
# Manual upscaling factors (applied before detection).
SCALES = [1.0, 1.5, 2.0, 3.0]

# A predicted identity is accepted only if nearest gallery distance <= threshold.
MATCH_THRESHOLD = 0.35

# Use up to 20 gallery images per identity for training/evaluation.
MAX_GALLERY_PER_ID = 20
# Set to an integer (e.g., 5) for quick debugging runs.
MAX_SCENES = None

# Fail early if dataset is missing.
assert DATASET_ROOT.exists(), f"Dataset folder not found: {DATASET_ROOT}"

In [ ]:
# -----------------------------
# Helper functions
# -----------------------------
def list_images(folder: Path):
    """Return all supported image files inside a folder."""
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])


def infer_single_identity(path: Path) -> str:
    """
    For gallery files, identity is taken from filename prefix.
    Example: 'c_12.jpg' -> 'C'.
    """
    token = path.stem.split("_")[0]
    return token[:1].upper()


def infer_scene_identities(path: Path):
    """
    For scene files, filename token may contain multiple people.
    Example: 'cd_gp_0_eo_12.jpg' -> {'C', 'D'}.
    """
    token = path.stem.split("_")[0].upper()
    ids = set(re.findall(r"[A-Z]", token))
    return ids


def cosine_distance(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    """Cosine distance (smaller means more similar)."""
    a_norm = np.linalg.norm(a) + eps
    b_norm = np.linalg.norm(b) + eps
    return float(1.0 - np.dot(a, b) / (a_norm * b_norm))


def precision_recall_f1(expected_ids, predicted_ids):
    """
    Set-based metrics on identities per scene:
    - expected_ids: true identities present in scene
    - predicted_ids: identities predicted by pipeline
    """
    tp = len(expected_ids & predicted_ids)
    fp = len(predicted_ids - expected_ids)
    fn = len(expected_ids - predicted_ids)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


# -----------------------------
# Build gallery + scene tables
# -----------------------------
gallery_dir = DATASET_ROOT / "photos_all_faces"
scene_dir = DATASET_ROOT / "group_photos"

gallery_paths = list_images(gallery_dir)
scene_paths = list_images(scene_dir)

# Optional quick-run mode: keep only first N scenes.
if MAX_SCENES is not None:
    scene_paths = scene_paths[:MAX_SCENES]

# Gallery dataframe: one row per known identity image.
gallery_rows = []
for p in gallery_paths:
    gallery_rows.append({"path": str(p), "identity": infer_single_identity(p)})
gallery_df = pd.DataFrame(gallery_rows)

# Enforce max images per identity to control runtime and class balance.
gallery_df = gallery_df.groupby("identity", group_keys=False).head(MAX_GALLERY_PER_ID).reset_index(drop=True)

# Scene dataframe: each row includes expected IDs (ground truth).
scene_rows = []
for p in scene_paths:
    expected = infer_scene_identities(p)
    if expected:
        scene_rows.append({"path": str(p), "expected_ids": expected})
scene_df = pd.DataFrame(scene_rows)

print(f"Gallery images used: {len(gallery_df)}")
print(f"Scene images used: {len(scene_df)}")
print("Expected scene IDs example:", scene_df.iloc[0]["expected_ids"] if len(scene_df) else set())

In [ ]:
# -----------------------------
# Build gallery embedding cache
# -----------------------------
# Why cache?
# Every combination reuses the same gallery, but each recognizer has different
# embedding space. So we compute gallery embeddings once per recognizer model,
# then reuse during combination scoring.
gallery_cache = {}

for model_name in RECOGNIZERS:
    rows = []
    print(f"Building gallery cache for model: {model_name}")

    # Iterate through known gallery faces and extract embeddings.
    for _, row in tqdm(gallery_df.iterrows(), total=len(gallery_df), desc=f"gallery:{model_name}"):
        try:
            # detector_backend='skip' because gallery images are already face crops.
            reps = DeepFace.represent(
                img_path=row["path"],
                model_name=model_name,
                detector_backend="skip",
                enforce_detection=False,
            )
            if len(reps) == 0:
                continue

            emb = np.array(reps[0]["embedding"], dtype=np.float32)
            rows.append({
                "identity": row["identity"],
                "embedding": emb,
                "path": row["path"],
            })
        except Exception:
            # Keep loop robust if one image/model call fails.
            continue

    gallery_cache[model_name] = pd.DataFrame(rows)
    print(f"  cached embeddings: {len(gallery_cache[model_name])}")

In [ ]:
# -----------------------------
# Combination scoring pipeline
# -----------------------------
def detect_faces_scaled(scene_bgr, detector_backend: str, scale: float):
    """
    Detect faces after optional upscaling.
    Upscaling can help some detectors find smaller faces.
    """
    if scale != 1.0:
        resized = cv2.resize(scene_bgr, dsize=None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    else:
        resized = scene_bgr

    faces = DeepFace.extract_faces(
        img_path=resized,
        detector_backend=detector_backend,
        enforce_detection=False,
        align=True,
    )
    return faces


# Each row in combo_rows will represent one (detector, recognizer, scale) tuple.
combo_rows = []
combo_list = list(itertools.product(DETECTORS, RECOGNIZERS, SCALES))

# Outer loop: test every combination exactly once.
for detector, recognizer, scale in tqdm(combo_list, desc="Combinations"):
    gallery_model_df = gallery_cache.get(recognizer, pd.DataFrame())
    if len(gallery_model_df) == 0:
        continue

    # scene_metrics collects per-scene results for this single combination.
    scene_metrics = []
    for _, scene_row in scene_df.iterrows():
        expected_ids = set(scene_row["expected_ids"])
        scene_bgr = cv2.imread(scene_row["path"])
        if scene_bgr is None:
            continue

        # 1) Detect scene faces with selected detector + scale.
        try:
            faces = detect_faces_scaled(scene_bgr, detector, scale)
        except Exception:
            faces = []

        # 2) Recognize each detected face by nearest gallery identity.
        predicted_ids = set()
        for face_item in faces:
            face_img = face_item.get("face", None)
            if face_img is None or np.size(face_img) == 0:
                continue
            try:
                # detector_backend='skip' because face_img is already a cropped face.
                rep = DeepFace.represent(
                    img_path=face_img,
                    model_name=recognizer,
                    detector_backend="skip",
                    enforce_detection=False,
                )
                if len(rep) == 0:
                    continue

                emb = np.array(rep[0]["embedding"], dtype=np.float32)

                # Compare this face embedding with all gallery embeddings.
                dists = gallery_model_df["embedding"].apply(lambda e: cosine_distance(emb, e))
                best_idx = int(dists.idxmin())
                best_dist = float(dists.loc[best_idx])

                # Accept identity only if distance is good enough.
                if best_dist <= MATCH_THRESHOLD:
                    predicted_ids.add(gallery_model_df.loc[best_idx, "identity"])
            except Exception:
                continue

        # 3) Evaluate predicted identity set vs expected set for this scene.
        precision, recall, f1 = precision_recall_f1(expected_ids, predicted_ids)
        scene_metrics.append({
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "num_predicted_ids": len(predicted_ids),
            "num_expected_ids": len(expected_ids),
        })

    if len(scene_metrics) == 0:
        continue

    # Aggregate this combination's performance over all scenes.
    mdf = pd.DataFrame(scene_metrics)
    combo_rows.append({
        "detector": detector,
        "recognizer": recognizer,
        "scale": scale,
        "mean_precision": mdf["precision"].mean(),
        "mean_recall": mdf["recall"].mean(),
        "mean_f1": mdf["f1"].mean(),
        "mean_num_predicted_ids": mdf["num_predicted_ids"].mean(),
    })

# Final score table sorted by ground-truth mean F1.
combo_df = pd.DataFrame(combo_rows).sort_values("mean_f1", ascending=False).reset_index(drop=True)
print(f"Valid combinations scored: {len(combo_df)}")
combo_df.head(10)

In [ ]:
# -----------------------------
# Simple ANN (NumPy implementation)
# -----------------------------
# Goal: learn a smooth mapping
#   (detector, recognizer, scale) -> expected quality (mean_f1)
# This gives a data-driven weighting instead of manual weighting.

# Create index maps for one-hot encoding categorical features.
detector_to_idx = {d: i for i, d in enumerate(DETECTORS)}
recognizer_to_idx = {r: i for i, r in enumerate(RECOGNIZERS)}

num_detector = len(DETECTORS)
num_recognizer = len(RECOGNIZERS)
# +1 for the numeric scale feature.
input_dim = num_detector + num_recognizer + 1

# X = encoded combination features, y = measured combo mean_f1.
X = np.zeros((len(combo_df), input_dim), dtype=np.float32)
y = combo_df["mean_f1"].values.astype(np.float32).reshape(-1, 1)

# Encode each combination row.
for i, row in combo_df.iterrows():
    # Detector one-hot
    X[i, detector_to_idx[row["detector"]]] = 1.0
    # Recognizer one-hot (offset by detector block length)
    X[i, num_detector + recognizer_to_idx[row["recognizer"]]] = 1.0
    # Scale normalized to [0, 1]
    X[i, -1] = row["scale"] / max(SCALES)

# Random split for lightweight train/test reporting.
rng = np.random.default_rng(42)
idx = np.arange(len(X))
rng.shuffle(idx)
split = max(1, int(0.8 * len(X)))
train_idx, test_idx = idx[:split], idx[split:]

X_train, y_train = X[train_idx], y[train_idx]
if len(test_idx) > 0:
    X_test, y_test = X[test_idx], y[test_idx]
else:
    # Handle tiny datasets safely.
    X_test = np.empty((0, input_dim), dtype=np.float32)
    y_test = np.empty((0, 1), dtype=np.float32)

# Network shape: input -> hidden(ReLU) -> output(linear)
hidden_dim = 12
W1 = rng.normal(0, 0.1, size=(input_dim, hidden_dim)).astype(np.float32)
b1 = np.zeros((1, hidden_dim), dtype=np.float32)
W2 = rng.normal(0, 0.1, size=(hidden_dim, 1)).astype(np.float32)
b2 = np.zeros((1, 1), dtype=np.float32)

def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(np.float32)

lr = 0.03
epochs = 1500

# Manual gradient-descent training loop (MSE loss).
for ep in range(epochs):
    # Forward pass
    z1 = X_train @ W1 + b1
    h1 = relu(z1)
    y_hat = h1 @ W2 + b2

    # Loss
    err = y_hat - y_train
    loss = float(np.mean(err ** 2))

    # Backward pass
    n = len(X_train)
    d_yhat = (2.0 / max(1, n)) * err
    dW2 = h1.T @ d_yhat
    db2 = np.sum(d_yhat, axis=0, keepdims=True)
    dh1 = d_yhat @ W2.T
    dz1 = dh1 * relu_grad(z1)
    dW1 = X_train.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    # Parameter update
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

    if ep % 300 == 0 or ep == epochs - 1:
        print(f"epoch={ep:4d} train_mse={loss:.6f}")


def predict_ann(X_in):
    """Inference helper for ANN-predicted combination score."""
    return relu(X_in @ W1 + b1) @ W2 + b2

# Report basic fit quality.
train_pred = predict_ann(X_train)
train_mse = float(np.mean((train_pred - y_train) ** 2))
print(f"\nTrain MSE: {train_mse:.6f}")

if len(X_test) > 0:
    test_pred = predict_ann(X_test)
    test_mse = float(np.mean((test_pred - y_test) ** 2))
    print(f"Test MSE:  {test_mse:.6f}")
else:
    print("Test set empty (very small number of valid combinations).")

In [ ]:
# -----------------------------
# Learned weightage extraction
# -----------------------------
# We estimate feature importance from the first layer weights.
# Intuition: if a feature has larger absolute weights on average,
# it influences hidden activations more strongly.
feature_importance = np.mean(np.abs(W1), axis=1)

# Split importances back into feature groups.
detector_importance = feature_importance[:num_detector]
recognizer_importance = feature_importance[num_detector:num_detector + num_recognizer]
scale_importance = feature_importance[-1]


def to_percentage(arr):
    """Normalize raw importances into percentages for readability."""
    arr = np.array(arr, dtype=np.float64)
    s = arr.sum()
    if s <= 0:
        return np.zeros_like(arr)
    return 100.0 * arr / s


# Weightage percentages within each categorical group.
detector_weight_pct = to_percentage(detector_importance)
recognizer_weight_pct = to_percentage(recognizer_importance)

# Build pretty tables for report output.
detector_weight_df = pd.DataFrame({
    "detector": DETECTORS,
    "weight_percent": detector_weight_pct,
}).sort_values("weight_percent", ascending=False)

recognizer_weight_df = pd.DataFrame({
    "recognizer": RECOGNIZERS,
    "weight_percent": recognizer_weight_pct,
}).sort_values("weight_percent", ascending=False)

print("Detector weightage (learned):")
display(detector_weight_df)

print("Recognizer weightage (learned):")
display(recognizer_weight_df)

# Scale is a single numeric feature, so we show its raw magnitude.
print(f"Scale feature importance (raw): {scale_importance:.6f}")

# ANN-based ranking: this is the model's predicted quality per combo.
combo_df = combo_df.copy()
combo_df["ann_predicted_f1"] = predict_ann(X).reshape(-1)
combo_df = combo_df.sort_values("ann_predicted_f1", ascending=False).reset_index(drop=True)

print("Top combinations by ANN-predicted F1:")
combo_df.head(15)